# STEP 1 — Load Datasets & Verify BM25 Retriever

**Prerequisite:** Step 0 complete. `slm-agents-code`, `slm-agents-models` attached as inputs.

**What it does:**
1. Initializes the session (restores results from prior sessions if any)
2. Downloads HotpotQA + MuSiQue
3. Verifies BM25 retrieval quality on sample questions
4. Saves datasets to `data/` for all subsequent steps
5. Calls session_save at the end

**Outputs:** `data/hotpotqa.json`, `data/musique.json`

**Run time:** ~10 minutes

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/slm_agents_paper')
from src.session_init import init
P = init()

In [ ]:
!pip install -q rank_bm25 datasets transformers accelerate bitsandbytes scikit-learn

In [ ]:
from src.datasets import load_hotpotqa, load_musique, save_samples

hotpotqa = load_hotpotqa(n=300)
musique  = load_musique(n=200)

save_samples(hotpotqa, P['data'] / 'hotpotqa.json')
save_samples(musique,  P['data'] / 'musique.json')

print(f'HotpotQA: {len(hotpotqa)} | MuSiQue: {len(musique)}')

In [ ]:
# Verify BM25 retrieval on 5 samples
from src.datasets import build_bm25_index, retrieve

for sample in hotpotqa[:5]:
    index  = build_bm25_index(sample['passages'])
    result = retrieve(sample['question'], index, sample['passages'], top_k=1)
    print(f"Q: {sample['question'][:80]}")
    print(f"A: {sample['answer']}")
    print(f"Retrieved: {result[:150]}")
    print()

In [ ]:
# Dataset stats
import statistics
hp_steps_est = [len(s['passages']) for s in hotpotqa]
mu_steps_est = [len(s['passages']) for s in musique]
print(f'HotpotQA avg passages/q: {statistics.mean(hp_steps_est):.1f}')
print(f'MuSiQue  avg passages/q: {statistics.mean(mu_steps_est):.1f}')

# Answer length distribution
hp_ans_len = [len(s['answer'].split()) for s in hotpotqa]
print(f'HotpotQA avg answer words: {statistics.mean(hp_ans_len):.1f}')

print('\n✅ STEP 1 COMPLETE')

In [ ]:
from src.session_save import save
save(P['results'])
print('Upload results_snapshot_*.zip as new version of slm-agents-results before closing')